In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv(r"original datasets\UCI_Credit_Card.csv")
bill_cols = [c for c in df.columns if c.startswith('BILL_AMT')]
pay_amt_cols = [c for c in df.columns if c.startswith('PAY_AMT')]

No missing values found.

In [3]:
cols = df.columns.drop("ID")

inactive = (df[bill_cols + pay_amt_cols] == 0).all(axis=1)
extra_copy = df.duplicated(subset=cols, keep="first")

to_drop = extra_copy & ~inactive
df_clean = df[~to_drop].copy()

print(to_drop.sum(), len(df_clean))   

5 29995


**Duplicates:** IDs are unique, but 35 pairs match on every column except ID.
30 pairs have all-zero bills and payments (inactive accounts), so a match could be coincidence. These are kept.
5 pairs have non-trivial histories, where an exact match is very unlikely by chance. Assumed to be true duplicates; one copy of each pair is dropped.
After dropping, 29995 rows remain.

In [4]:
print(df_clean["EDUCATION"].value_counts().sort_index())
print(df_clean["MARRIAGE"].value_counts().sort_index())

EDUCATION
0       14
1    10584
2    14027
3     4916
4      123
5      280
6       51
Name: count, dtype: int64
MARRIAGE
0       54
1    13656
2    15962
3      323
Name: count, dtype: int64


In the education column, values 0,5,6 are undefined in the documentation. 
Since 4 is labelled as 'others', we can merge the undefined categories with 'others' as they both do not belong to any of the defined categories. 
Same thing with value 0 in the marriage column. Merging it with 3, which is the 'others' category is an option.

This is a better option than dropping these values since they account for around 400 rows (2%).

In [5]:
df_clean["EDUCATION"] = df_clean["EDUCATION"].replace({0: 4, 5: 4, 6: 4})
df_clean["MARRIAGE"] = df_clean["MARRIAGE"].replace({0: 3})
print(df_clean["EDUCATION"].value_counts().sort_index())
print(df_clean["MARRIAGE"].value_counts().sort_index())

EDUCATION
1    10584
2    14027
3     4916
4      468
Name: count, dtype: int64
MARRIAGE
1    13656
2    15962
3      377
Name: count, dtype: int64


Now the values belong in the defined range.

**Undocumented codes:**
- EDUCATION 0, 5, 6 (335 rows) merged into 4 ("others"). No rows lost; the codes are undefined but the rest of the row is valid.
- MARRIAGE 0 (54 rows) merged into 3 ("others"), same reasoning.
- PAY_ codes -2 and 0 kept unchanged (>50% of values). Treated as ordinal; meaning is an assumption since they're not in the documentation.

In [6]:
print(df_clean[["AGE", "LIMIT_BAL"]].describe())
print((df_clean["AGE"] < 18).sum(), (df_clean["LIMIT_BAL"] <= 0).sum())
print((df_clean[bill_cols] < 0).sum())
print((df_clean[pay_amt_cols] < 0).sum())

                AGE       LIMIT_BAL
count  29995.000000    29995.000000
mean      35.486281   167485.570262
std        9.218064   129754.919486
min       21.000000    10000.000000
25%       28.000000    50000.000000
50%       34.000000   140000.000000
75%       41.000000   240000.000000
max       79.000000  1000000.000000
0 0
BILL_AMT1    590
BILL_AMT2    669
BILL_AMT3    655
BILL_AMT4    675
BILL_AMT5    655
BILL_AMT6    688
dtype: int64
PAY_AMT1    0
PAY_AMT2    0
PAY_AMT3    0
PAY_AMT4    0
PAY_AMT5    0
PAY_AMT6    0
dtype: int64


**Value validation:**
- AGE ranges 21 to 79: plausible, no changes.
- LIMIT_BAL ranges 10,000 to 1,000,000, no zero or negative values. Right-skewed (mean 167k > median 140k). Large limits kept as legitimate high-value customers; outlier treatment is handled in Step 5.
- BILL_AMT has about 590 to 690 negative values per month (~2%). Kept: a negative bill can mean overpayment or a refund credit, and the consistent count across months suggests genuine values rather than errors.
- PAY_AMT has no negative values.
- Zeros in BILL_AMT / PAY_AMT treated as valid (unused card or no payment), not missing.

In [9]:
df_clean.to_csv(r"processed datasets/cleaned_UCI_Credit_Card.csv", index=False)
print(df_clean.shape)

(29995, 25)


## Cleaning Summary

| # | Issue | Found | Decision | Reason | Rows affected |
|---|---|---|---|---|---|
| 1 | Duplicate customers | 35 pairs identical except ID | Dropped 1 copy from the 5 non-trivial pairs; kept 30 inactive pairs | Specific histories can't match by chance; all-zero rows can | 5 dropped |
| 2 | EDUCATION codes 0, 5, 6 | undocumented | Merged into 4 (others) | Row is otherwise valid | 335 recoded |
| 3 | MARRIAGE code 0 | undocumented | Merged into 3 (others) | Same reasoning | 54 recoded |
| 4 | PAY_ codes -2, 0 | undocumented, >50% of values | Kept unchanged | Too many to drop; treated as ordinal "no delay" states (assumption) | 0 |
| 5 | Negative BILL_AMT | ~590-690 per month | Kept | Likely overpayment or credit; can't verify as error | 0 |
| 6 | LIMIT_BAL / AGE extremes | max 1,000,000; age 21-79 | Kept | Plausible; outliers handled in Step 5 | 0 |
| 7 | Missing values | none | None needed | | 0 |

Final shape: 29,995 rows x 25 columns, saved to processed datasets\cleaned_UCI_Credit_Card.csv